In [18]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/1976-2024-house.tab")

# some cols have both True and "True" values
bool_cols = ["runoff", "special", "writein", "unofficial", "fusion_ticket"]
for col in bool_cols:
    df[col] = df[col].astype(str).str.strip().str.upper() == "TRUE"

print(df.shape)
print(df.head(2))

(33805, 20)
   year    state state_po  state_fips  state_cen  state_ic    office  \
0  1976  ALABAMA       AL           1         63        41  US HOUSE   
1  1976  ALABAMA       AL           1         63        41  US HOUSE   

   district stage  runoff  special       candidate       party  writein  \
0         1   GEN   False    False  BILL DAVENPORT    DEMOCRAT    False   
1         1   GEN   False    False    JACK EDWARDS  REPUBLICAN    False   

    mode  candidatevotes  totalvotes  unofficial   version  fusion_ticket  
0  TOTAL           58906      157170       False  20250910          False  
1  TOTAL           98257      157170       False  20250910          False  


C:\Users\Таня\AppData\Local\Temp\ipykernel_3304\3437170378.py:4: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/1976-2024-house.tab")


Alabama is roughly 7 times bigger than Alaska population-wise and that's why there are 7x districts. At-large convention in the MIT dataset: states with a single at-large district code it as 0, while states with multiple districts number them starting from 1


In [19]:
print(
    df[(df["state_po"] == "AL") & (df["year"] == 2024) & (df["stage"] == "GEN")][
        "district"
    ].unique()
)
print(
    df[(df["state_po"] == "AK") & (df["year"] == 2024) & (df["stage"] == "GEN")][
        "district"
    ].unique()
)

[1 2 3 4 5 6 7]
[0]


### Drop useless columns


In [20]:
for col in df.columns:
    print(f"\n--- {col} ({df[col].nunique()} unique values) ---")
    print(df[col].value_counts().head(10))

# first three are just a different way to encode state_po
# fusion_ticket is a labeling quirk (mainly NY), doesn't change who won
useless_cols = ["state_fips", "state_cen", "state_ic", "fusion_ticket"]

# these cols have just one value - drop them
cols_with_one_value = ["office", "mode", "version"]

# these cols have two values, filter out the minor category, then drop the col
# stage == PRI в†’ intra-party contests (e.g. two Democrats fighting with each other)
# special == TRUE в†’ special elections (vacancy mid-term), different dynamics than regular elections
# runoff == TRUE в†’ a second round after no one hit a threshold, mostly a Southern thing
# unofficial == TRUE в†’ uncertified results
# writein == TRUE в†’ filter out, write-in candidates never win and distort vote totals
filter_then_drop = ["stage", "special", "runoff", "unofficial", "writein"]


--- year (25 unique values) ---
1996    1660
2000    1608
1992    1524
2010    1500
2020    1467
2016    1440
2002    1433
2012    1427
2008    1381
2006    1374
Name: year, dtype: int64

--- state (51 unique values) ---
NEW YORK        4137
CALIFORNIA      3447
TEXAS           2089
MICHIGAN        1641
NEW JERSEY      1332
FLORIDA         1290
ILLINOIS        1261
PENNSYLVANIA    1232
OHIO            1192
VIRGINIA         878
Name: state, dtype: int64

--- state_po (51 unique values) ---
NY    4137
CA    3447
TX    2089
MI    1641
NJ    1332
FL    1290
IL    1261
PA    1232
OH    1192
VA     878
Name: state_po, dtype: int64

--- state_fips (51 unique values) ---
36    4137
6     3447
48    2089
26    1641
34    1332
12    1290
17    1261
42    1232
39    1192
51     878
Name: state_fips, dtype: int64

--- state_cen (51 unique values) ---
21    4137
93    3447
74    2089
34    1641
22    1332
59    1290
33    1261
23    1232
31    1192
54     878
Name: state_cen, dtype: int64

--- sta

In [21]:
if "stage" in df.columns:
    df = df[df["stage"] == "GEN"]

for col in filter_then_drop[1:]:
    if col in df.columns:
        df = df[df[col] == False]

cols_to_drop = [
    c for c in useless_cols + cols_with_one_value + filter_then_drop if c in df.columns
]
df = df.drop(columns=cols_to_drop).reset_index(drop=True).copy()

print(df.shape)
print(df.columns.tolist())

(30824, 8)
['year', 'state', 'state_po', 'district', 'candidate', 'party', 'candidatevotes', 'totalvotes']


### Filter out NA values


In [22]:
print(df.isnull().sum())
df[df["party"].isnull()]["candidate"].value_counts()

year                 0
state                0
state_po             0
district             0
candidate            0
party             1290
candidatevotes       0
totalvotes           0
dtype: int64


OTHER                              420
BLANK VOTE/SCATTERING              373
SCATTERING                          68
BLANK VOTE                          65
VOID                                55
BLANK                               54
UNDERVOTES                          54
BLANK VOTE/VOID VOTE/SCATTERING     54
BLANKS                              18
ALL OTHERS                          18
OVERVOTES                           18
OVER VOTE                           16
VOID VOTE                           16
MISCELLANEOUS                       12
OVER VOTES                           9
OTHER WRITE-INS                      8
BLANK VOTES                          6
UNDER VOTES                          6
WORKING FAMILIES                     4
INDEPENDENT                          3
SCATTER                              2
ERNEST JOHNSON                       1
RANDY ALTSCHULER                     1
DAN HILL                             1
MARTIN JAMES MONROE                  1
MODERATE                 

In [23]:
# Even the real ones (INDEPENDENT, WORKING FAMILIES or real names) almost never win House seats,
# so they won't affect a D vs R prediction significantly
df = df[df["party"].notna()].reset_index(drop=True)
print(df.shape)
print(df.dtypes)

(29534, 8)
year               int64
state             object
state_po          object
district           int64
candidate         object
party             object
candidatevotes     int64
totalvotes         int64
dtype: object


In [24]:
df.describe()

,year,district,candidatevotes,totalvotes
count,29534.000000,29534.000000,29534.000000,29534.000000
mean,2000.363513,10.075201,77488.540868,218320.248324
std,14.172550,10.212681,66151.552844,79923.592746
min,1976.000000,0.000000,-1.000000,-1.000000
25%,1988.000000,3.000000,8678.250000,163389.500000
50%,2000.000000,6.000000,72515.500000,208776.000000
75%,2012.000000,14.000000,121843.250000,268524.000000
max,2024.000000,53.000000,387109.000000,656104.000000


In [25]:
df = df[(df["candidatevotes"] >= 0) & (df["totalvotes"] >= 0)].reset_index(drop=True)
print(df.shape)
print(sorted(df["year"].unique()))

party_map = {
    "DEMOCRATIC-FARMER-LABOR": "DEMOCRAT",
    "FOGLIETTA (DEMOCRAT)": "DEMOCRAT",
    "INDEPENDENT-REPUBLICAN": "REPUBLICAN",
}
df["party"] = df["party"].replace(party_map)

# keep only the top candidate per party per race вЂ” correct handling of rare multi-candidate cases
party_votes = df.loc[
    df.groupby(["year", "state_po", "district", "party"])["candidatevotes"].idxmax()
][
    ["year", "state_po", "district", "party", "candidatevotes", "totalvotes"]
].reset_index(
    drop=True
)

reps = party_votes[party_votes["party"] == "REPUBLICAN"][
    ["year", "state_po", "district", "candidatevotes"]
].rename(columns={"candidatevotes": "rep_votes"})
dems = (
    party_votes[party_votes["party"] == "DEMOCRAT"]
    .rename(columns={"candidatevotes": "dem_votes"})
    .copy()
)
dems_reps = dems.merge(reps, on=["year", "state_po", "district"], how="left")
# if no rep, NA will be the value for rep_votes, changing it to zero makes predictions worse
dems_reps["dem_vote_share"] = dems_reps["dem_votes"] / (
    dems_reps["dem_votes"] + dems_reps["rep_votes"]
)

dems_reps = dems_reps.sort_values(["state_po", "district", "year"])
# 0.5 -> no lean, <0.3 heavy rep_lean, >0.7 heavy dem_lean
# window=4 (8 years) captures structural partisan identity better than window=2
dems_reps["dem_lean"] = dems_reps.groupby(["state_po", "district"])[
    "dem_vote_share"
].transform(lambda x: x.shift(1).rolling(4).mean())
print(
    dems_reps.shape
)  # typical election: 1 dem, 1 rep, 1 other (most often a libertarian)

(29531, 8)
[1976, 1978, 1980, 1982, 1984, 1986, 1988, 1990, 1992, 1994, 1996, 1998, 2000, 2002, 2004, 2006, 2008, 2010, 2012, 2014, 2016, 2018, 2020, 2022, 2024]
(10281, 9)


### Find winners


In [26]:
approval = pd.read_csv(
    "../data/Approval-Ratings-for-POTUS-raw.xls",
    sep="\t",
    header=None,
    names=["start", "end", "approve", "disapprove", "unsure"],
)
approval["year"] = pd.to_datetime(approval["start"]).dt.year
approval = approval[["year", "approve", "disapprove"]].rename(
    columns={"approve": "pres_approval", "disapprove": "pres_disapproval"}
)

pres_party = {
    1976: "REPUBLICAN",  # Ford
    1978: "DEMOCRAT",  # Carter
    1980: "DEMOCRAT",  # Carter
    1982: "REPUBLICAN",  # Reagan
    1984: "REPUBLICAN",
    1986: "REPUBLICAN",
    1988: "REPUBLICAN",
    1990: "REPUBLICAN",  # Bush Sr
    1992: "REPUBLICAN",
    1994: "DEMOCRAT",  # Clinton
    1996: "DEMOCRAT",
    1998: "DEMOCRAT",
    2000: "DEMOCRAT",
    2002: "REPUBLICAN",  # Bush Jr
    2004: "REPUBLICAN",
    2006: "REPUBLICAN",
    2008: "REPUBLICAN",
    2010: "DEMOCRAT",  # Obama
    2012: "DEMOCRAT",
    2014: "DEMOCRAT",
    2016: "DEMOCRAT",
    2018: "REPUBLICAN",  # Trump
    2020: "REPUBLICAN",
    2022: "DEMOCRAT",  # Biden
    2024: "DEMOCRAT",
    2026: "REPUBLICAN",  # Trump
}
approval["pres_party"] = approval["year"].map(pres_party)

# flip sign so higher always means "better for Democrats":
# dem pres: high approval = good for Dems в†’ keep positive
# rep pres: high approval = bad for Dems в†’ flip negative
sign = np.where(approval["pres_party"] == "DEMOCRAT", 1, -1)
approval["pres_approval"] = approval["pres_approval"] * sign
approval["pres_disapproval"] = approval["pres_disapproval"] * (-sign)

approval = approval[["year", "pres_approval", "pres_disapproval"]]
print(approval)

    year  pres_approval  pres_disapproval
0   1976            -45                40
1   1978             49               -36
2   1980             37               -55
3   1982            -42                48
4   1984            -58                33
5   1986            -63                29
6   1988            -51                38
7   1990            -58                32
8   1992            -34                56
9   1994             46               -46
10  1996             54               -36
11  1998             66               -30
12  2000             57               -38
13  2002            -63                29
14  2004            -48                47
15  2006            -38                56
16  2008            -25                70
17  2010             45               -48
18  2012             52               -45
19  2014             42               -53
20  2016             53               -45
21  2018            -40                54
22  2020            -46           

In [27]:
winners = party_votes.loc[
    party_votes.groupby(["year", "state_po", "district"])["candidatevotes"].idxmax()
].reset_index(drop=True)
winners["vote_share"] = winners["candidatevotes"] / winners["totalvotes"]
winners = winners.merge(
    dems_reps[["year", "state_po", "district", "dem_lean"]],
    on=["year", "state_po", "district"],
    how="left",
)
winners = winners.merge(approval, on="year", how="left")

# Mostly Bernie Sanders
print(
    winners[~winners["party"].isin(["DEMOCRAT", "REPUBLICAN"])][
        ["year", "state_po", "district", "party"]
    ].value_counts()
)

print(winners.shape)
# NaNs come from first two elections, rep-only and new districts with no history
print(winners["dem_lean"].isna().sum())
print(winners.head())

year  state_po  district  party      
1990  VT        0         INDEPENDENT    1
1992  VT        0         INDEPENDENT    1
1994  VT        0         INDEPENDENT    1
1996  MO        8         INDEPENDENT    1
      VT        0         INDEPENDENT    1
1998  VT        0         INDEPENDENT    1
2000  VA        5         INDEPENDENT    1
      VT        0         INDEPENDENT    1
2002  VT        0         INDEPENDENT    1
2004  VT        0         INDEPENDENT    1
dtype: int64
(10841, 10)
4355
   year state_po  district       party  candidatevotes  totalvotes  \
0  1976       AK         0  REPUBLICAN           83722      118208   
1  1976       AL         1  REPUBLICAN           98257      157170   
2  1976       AL         2  REPUBLICAN           90069      156362   
3  1976       AL         3    DEMOCRAT          106935      108048   
4  1976       AL         4    DEMOCRAT          141490      176022   

   vote_share  dem_lean  pres_approval  pres_disapproval  
0    0.708260       Na

### Feature generation


In [28]:
# MIT data has ~3% gaps for years 1996 and 2018
seats_per_year = (
    winners.groupby(["year", "party"])["district"].count().unstack(fill_value=0)
)
seats_per_year = seats_per_year[["DEMOCRAT", "REPUBLICAN"]]
seats_per_year["Total"] = seats_per_year["DEMOCRAT"] + seats_per_year["REPUBLICAN"]
seats_per_year["Majority"] = np.where(
    seats_per_year["DEMOCRAT"] > seats_per_year["REPUBLICAN"], "DEMOCRAT", "REPUBLICAN"
)

print(
    len(seats_per_year)
)  # some years weren't there because of the boolean col bug (see cell 1)
print(seats_per_year)

25
party  DEMOCRAT  REPUBLICAN  Total    Majority
year                                          
1976        291         144    435    DEMOCRAT
1978        277         158    435    DEMOCRAT
1980        245         190    435    DEMOCRAT
1982        270         165    435    DEMOCRAT
1984        254         181    435    DEMOCRAT
1986        259         176    435    DEMOCRAT
1988        260         175    435    DEMOCRAT
1990        267         167    434    DEMOCRAT
1992        258         176    434    DEMOCRAT
1994        205         229    434  REPUBLICAN
1996        203         217    420  REPUBLICAN
1998        211         223    434  REPUBLICAN
2000        213         220    433  REPUBLICAN
2002        205         229    434  REPUBLICAN
2004        202         232    434  REPUBLICAN
2006        232         199    431    DEMOCRAT
2008        257         178    435    DEMOCRAT
2010        197         238    435  REPUBLICAN
2012        202         233    435  REPUBLICAN
2014      

In [29]:
# Incumbency advantage вЂ” incumbents consistently outperform challengers by a few points across all eras
# nationwide
shifted = seats_per_year["Majority"].shift(1)
majority_map = shifted.to_dict()
winners["nation_incumbent_party"] = winners["year"].map(majority_map)

# district
winners = winners.sort_values(["state_po", "district", "year"])
winners["district_incumbent_party"] = winners.groupby(["state_po", "district"])[
    "party"
].shift(1)

winners["dem_won"] = (winners["party"] == "DEMOCRAT").astype(int)
winners = winners[winners["district_incumbent_party"] != "INDEPENDENT"]

winners = pd.get_dummies(
    winners,
    columns=["nation_incumbent_party", "district_incumbent_party"],
    drop_first=True,
)
print(winners.columns.tolist())

['year', 'state_po', 'district', 'party', 'candidatevotes', 'totalvotes', 'vote_share', 'dem_lean', 'pres_approval', 'pres_disapproval', 'dem_won', 'nation_incumbent_party_REPUBLICAN', 'district_incumbent_party_REPUBLICAN']


In [30]:
drop_cols = [
    "year",  # time index, not a generalizable signal
    "state_po",  # district identity, not a feature
    "district",  # district identity, not a feature
    "party",  # this is what we're predicting (via dem_won)
    "candidatevotes",  # current election result вЂ” data leakage
    "totalvotes",  # current election result вЂ” data leakage
    "vote_share",  # current election result вЂ” data leakage
    "dem_won",  # this is y
]

X = winners.drop(columns=drop_cols)
y = winners["dem_won"]

print(X.shape)
print(X.columns.tolist())
print(X.isnull().sum())

(10831, 5)
['dem_lean', 'pres_approval', 'pres_disapproval', 'nation_incumbent_party_REPUBLICAN', 'district_incumbent_party_REPUBLICAN']
dem_lean                               4351
pres_approval                             0
pres_disapproval                          0
nation_incumbent_party_REPUBLICAN         0
district_incumbent_party_REPUBLICAN       0
dtype: int64


In [31]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression

for w in range(2, 8):
    dl = dems_reps.copy()
    dl["dem_lean"] = dl.groupby(["state_po", "district"])["dem_vote_share"].transform(
        lambda x: x.shift(1).rolling(w).mean()
    )
    wt = party_votes.loc[
        party_votes.groupby(["year", "state_po", "district"])["candidatevotes"].idxmax()
    ].reset_index(drop=True)
    wt["vote_share"] = wt["candidatevotes"] / wt["totalvotes"]
    wt = wt.merge(
        dl[["year", "state_po", "district", "dem_lean"]],
        on=["year", "state_po", "district"],
        how="left",
    )
    wt = wt.merge(approval, on="year", how="left")
    wt["nation_incumbent_party"] = wt["year"].map(majority_map)
    wt = wt.sort_values(["state_po", "district", "year"])
    wt["district_incumbent_party"] = wt.groupby(["state_po", "district"])[
        "party"
    ].shift(1)
    wt["dem_won"] = (wt["party"] == "DEMOCRAT").astype(int)
    wt = wt[wt["district_incumbent_party"] != "INDEPENDENT"]
    wt = pd.get_dummies(
        wt,
        columns=["nation_incumbent_party", "district_incumbent_party"],
        drop_first=True,
    )
    Xw = wt.drop(columns=drop_cols).dropna()
    yw = wt["dem_won"][Xw.index]
    pipe = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression())])
    scores = cross_val_score(pipe, Xw, yw, cv=5)
    print(f"window={w}: {scores.mean():.4f}  (n={len(Xw)})")

window=2: 0.9086  (n=7986)
window=3: 0.9101  (n=7172)
window=4: 0.9133  (n=6480)
window=5: 0.9109  (n=5869)
window=6: 0.9084  (n=5308)
window=7: 0.9051  (n=4792)


In [32]:
X = X.dropna()
y = y[X.index]

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression

pipeline = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression())])
scores = cross_val_score(pipeline, X, y, cv=5)
print(scores)
print(scores.mean())

[0.91049383 0.90740741 0.9212963  0.90046296 0.92669753]
0.9132716049382716


In [33]:
pipeline = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression())])
pipeline.fit(X, y)

Pipeline(steps=[('scaler', StandardScaler()), ('model', LogisticRegression())])

In [34]:
# who held each district going into 2026
incumbents_2026 = winners[winners["year"] == 2024][
    ["state_po", "district", "party"]
].rename(columns={"party": "district_incumbent_party"})

# dem lean for 2026 = average of last 4 elections (matching training window)
dem_lean_2026 = (
    dems_reps[dems_reps["year"].isin([2018, 2020, 2022, 2024])]
    .groupby(["state_po", "district"])["dem_vote_share"]
    .mean()
    .reset_index()
    .rename(columns={"dem_vote_share": "dem_lean"})
)

pred_2026 = incumbents_2026.merge(
    dem_lean_2026, on=["state_po", "district"], how="left"
)
pred_2026["nation_incumbent_party"] = "REPUBLICAN"
pred_2026["pres_approval"] = -32  # Trump (Rep) вЂ” sign-flipped per convention
pred_2026["pres_disapproval"] = 65  # Trump (Rep) вЂ” high disapproval = Dem-favorable

pred_2026 = pd.get_dummies(
    pred_2026,
    columns=["nation_incumbent_party", "district_incumbent_party"],
    drop_first=True,
)
pred_2026 = pred_2026.reindex(columns=X.columns, fill_value=0)

# fill missing dem_lean with median (districts where no Democrat ran in any of the last 4 elections)
pred_2026["dem_lean"] = pred_2026["dem_lean"].fillna(pred_2026["dem_lean"].median())

predictions = pipeline.predict(pred_2026)
print(f"Democrat seats: {predictions.sum()}")
print(f"Republican seats: {len(predictions) - predictions.sum()}")

model = pipeline.named_steps["model"]

coef_abs = np.abs(model.coef_[0])
weights = coef_abs / coef_abs.sum() * 100

for feature, weight in sorted(
    zip(X.columns, weights), key=lambda x: x[1], reverse=True
):
    print(f"{feature}: {weight:.1f}%")

Democrat seats: 215
Republican seats: 219
district_incumbent_party_REPUBLICAN: 46.1%
dem_lean: 23.3%
pres_disapproval: 17.9%
pres_approval: 6.5%
nation_incumbent_party_REPUBLICAN: 6.2%


- `district_incumbent_party_REPUBLICAN` is an SMA(1) shifted by 1, _whoever was at the office last time_ is the biggest predictive factor whether they will be there again
- `dem_lean` is an SMA(8), shows a 16-year trend of elections in this district
- `pres_disapproval` > `pres_approval` - people tend to be more vocal against someone they hate rather than be supportive to someone they like
- `nation_incumbent_party_REPUBLICAN` - least important, the district (clan) previous results are way more important than the overall nation tendencies
